<a href="https://colab.research.google.com/github/lee-woonju/study-hongong-mldl/blob/main/05_3_%ED%8A%B8%EB%A6%AC%EC%9D%98_%EC%95%99%EC%83%81%EB%B8%94.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## KEYWORD : 앙상블 학습, 랜덤 포레스트, 엑스트라 트리, 그레이디언트 부스팅

# 정형 데이터와 비정형 데이터
- 정형데이터 : 어떤 구조로 되어있는 데이터. csv, 데이터베이스, 엑셀
- 비정형데이터 : 텍스트데이터(책의 글), 사진, 음악 등

### 앙상블 학습
- 일반적으로 정형 데이터를 다루는 데 가장 뛰어난 성과를 내는 알고리즘
- 대부분 **결정 트리**를 기반으로 만들어져 있음

### 신경망 알고리즘
- 비정형데이터는 규칙성을 찾기 어려워 전통적인 머신러닝 방법으로 모델을 만들기 까다로움

---

# 앙상블1. 랜덤 포레스트
- 대표적인 앙상블 학습
> 결정트리를 랜덤하게 만들어 결정 트리 숲을 만들고, 각 결정 트리의 예측을 사용해 최종 예측을 만듦

## 랜덤 포레스트 훈련 방식
- 각 트리를 훈련하기 위해 부트스트랩 샘플 만들기
> 부트스트랩 샘플 : 데이터 세트에서 중복을 허용하여 다시 샘플링하는 것. 기본적으로 훈련 세트의 크기와 같게 만듦
- 각 노드를 분할할 때 전체 특성 중 일부 특성을 무작위로 고르기
  - 분류모델 RandomForestClassifier 는 전체 특성 개수의 제곱근만큼 특성 선택
  - 회귀모델 RandomForestRegressor 은 전체 특성 사용
- 랜덤 포레스트는 기본적으로 100개의 결정 트리를 이런 방식으로 훈련
  - 분류일 때는 각 트리의 클래스별 확률을 평균하여 가장 높은 확률을 가진 클래스를 예측으로 삼음
  - 회귀일 때는 단순히 각 트리의 예측을 평균

### RandomForestClassifire



In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

wine = pd.read_csv('https://bit.ly/wine_csv_data')
data = wine[['alcohol', 'sugar', 'pH']]
target = wine['class']
train_input, test_input, train_target, test_target = train_test_split(data, target, test_size=0.2, random_state=42)

from sklearn.model_selection import cross_validate
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_jobs=-1, random_state=42)

# 교차 검증 수행 cross_validate
scores = cross_validate(rf, train_input, train_target, return_train_score=True, n_jobs=-1)
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

# 특성중요도 계산 (특성중요도 = 결정 트리의 큰 장점 중 하나)
rf.fit(train_input, train_target)
print(rf.feature_importances_)


0.9973541965122431 0.8905151032797809
[0.23167441 0.50039841 0.26792718]


랜덤 포레스트는 OOB샘플을 사용하여 부트스트랩 샘플로 훈련한 결정 트리를 평가할 수 있음
> OOB 샘플 ? 부트스트랩 샘플에 포함되지 않고 남는 샘플


In [7]:
rf = RandomForestClassifier(oob_score=True, n_jobs=-1, random_state=42)
rf.fit(train_input, train_target)
print(rf.oob_score_)

0.8934000384837406


# 앙상블2. 엑스트라 트리
- 랜덤 포레스트와의 차이는 **부트스트랩 샘플을 사용하지 않는다**는 점
- 각 결정 트리를 만들 때 전체 훈련 세트를 사용
- 대신 노드를 분할할 때 **무작위**로 분할



In [8]:
from sklearn.ensemble import ExtraTreesClassifier
et = ExtraTreesClassifier(n_jobs=-1, random_state=42)
scores = cross_validate(et, train_input, train_target, return_train_score=True, n_jobs=-1)
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.9974503966084433 0.8887848893166506


> 엑스트라 트리 : 랜덤하게 노드를 분할하기 때문에 계산 속도가 빠름

In [9]:
et.fit(train_input, train_target)
print(et.feature_importances_)

[0.20183568 0.52242907 0.27573525]


# 앙상블3. 그레이디언트 부스팅
> 깊이가 얕은 결정 트리를 사용하여 이전 트리의 오차를 보완하는 방식으로 앙상블 하는 방법


In [11]:
from sklearn.ensemble import GradientBoostingClassifier
gb = GradientBoostingClassifier(random_state=42)
scores = cross_validate(gb, train_input, train_target, return_train_score=True, n_jobs=-1)
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.8881086892152563 0.8720430147331015


> 그레이디언트 부스팅은 결정 트리의 개수를 늘려도 과대적합에 매우 강함

In [16]:
# n_estimators= 결정트리 개수
gb = GradientBoostingClassifier(n_estimators=500, learning_rate=0.2, random_state=42)
scores = cross_validate(gb, train_input, train_target, return_train_score=True, n_jobs=-1)
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

gb.fit(train_input, train_target)
print(gb.score(train_input, train_target))
print(gb.feature_importances_)


0.9464595437171814 0.8780082549788999
0.9382335963055609
[0.15887763 0.6799705  0.16115187]


# 앙상블4. 히스토그램 기반 그레이디언트 부스팅
> 정형 데이터를 다루는 머신러닝 알고리즘 중 가장 인기가 높음
- 데이터가 많아질수록 일반 그레이디언트 부스팅보다 훨씬 빠름
- 데이터를 256개의 구간으로 미리 범주화하기 때문에 빠름



In [19]:
from sklearn.ensemble import HistGradientBoostingClassifier
hgb = HistGradientBoostingClassifier(random_state=42)
scores = cross_validate(hgb, train_input, train_target, return_train_score=True)
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

# 특성 중요도 자체 제공하지 않음
from sklearn.inspection import permutation_importance
hgb.fit(train_input, train_target)

result = permutation_importance(hgb, train_input, train_target, n_repeats=10, random_state=42, n_jobs=-1)
print(result.importances_mean)

result = permutation_importance(hgb, test_input, test_target, n_repeats=10, random_state=42, n_jobs=-1)
print(result.importances_mean)

0.9321723946453317 0.8801241948619236
[0.08876275 0.23438522 0.08027708]
[0.05969231 0.20238462 0.049     ]


In [20]:
hgb.score(test_input, test_target)

0.8723076923076923

> 히스토그램 기반 그레이디언트 부스팅의 회귀버전은 HistGradientBoostingRegressor 클래스 말고도 여러가지가 있음

### **XGBoost** 라이브러리

In [21]:
from xgboost import XGBClassifier
xgb = XGBClassifier(tree_method='hist', random_state=42)
scores = cross_validate(xgb, train_input, train_target, return_train_score=True, n_jobs=-1)
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.9567059184812372 0.8783915747390243


### LightGBM

In [24]:
from lightgbm import LGBMClassifier
lgb = LGBMClassifier(random_state=42)
scores = cross_validate(lgb, train_input, train_target, return_train_score=True, n_jobs=-1)
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.935828414851749 0.8801251203079884


# [요약]
- 앙상블 : 하나의 강력한 모델을 만드는 대신, 여러 개의 평범한 모델을 학습시킨 뒤 그 결과를 합쳐서 더 정확한 예측을 하는 방법
- 얕은 트리 사용 $→$ 순차적 학습 $→$ 결합
> **랜덤 포레스트, 엑스트라 트리, 그레이디언트 부스팅, 히스토그램 기반 그레이디언트 부스팅** 을 배움

1. 랜덤 포레스트
    - 부트스트랩 샘플 사용, 가장 좋은 분할 찾음
    - 가장 대표적인 앙상블 학습 알고리즘, 성능이 좋고 안정적
2. 엑스트라 트리
    - 부트스트랩 샘플 사용 X, 결정 트리의 노드를 랜덤하게 분할
3. 그레이디언트 부스팅
    - 이전 트리의 손실을 보완하는 식으로 얕은 결정 트리를 연속하여 추가
4. 히스토그램 기반 그레이디언트 부스팅
    - 훈련 데이터를 256개의 정수 구간으로 나눠 빠르고 높은 성능
    - 가장 뛰어난 앙상블 학습으로 평가받음

> 지금까지는 입력과 타깃이 준비된 문제 $→$ 지도학습